# Time Series Regression with Signature Methods

This notebook demonstrates how to use the Nadaraya-Watson signature regressor for time series regression tasks and compares it with other baseline methods.

## Overview
- Load time series datasets from the Aeon library
- Augment time series with time dimension
- Apply signature-based regression
- Compare with DTW and signature-based baselines
- Evaluate model performance using MSE and R² metrics

## 1. Import Required Libraries

In [1]:
import sys
import os
# Add parent directory to path so we can import from nwsig package
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import matplotlib.pyplot as plt
import numpy as np
from aeon.datasets import load_regression
from aeon.regression.distance_based import KNeighborsTimeSeriesRegressor
from aeon.transformations.collection.signature_based import SignatureTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.utils import shuffle
from sklearn.preprocessing import MinMaxScaler

# Import from nwsig package
from nwsig import NWSignatureRegressor, augment_with_time

import warnings
warnings.filterwarnings('ignore')

## 2. Helper Functions

## 3. ARCH/GARCH

Let's start with a single dataset to understand the workflow.

In [2]:
import numpy as np
from arch import arch_model
from sklearn.model_selection import train_test_split

np.random.seed(0)
ret = np.random.randn(1000)

am = arch_model(ret, vol="Garch", p=1, q=1)
res = am.fit(disp="off")

n_sims = 100
n_obs = 100
simulations = np.zeros((n_sims, n_obs))

for i in range(n_sims):
    sim = am.simulate(res.params, nobs=n_obs)
    simulations[i, :] = sim['data']

X = simulations[:,:-1]
y = simulations[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training size: {X_train.shape}, Test size: {X_test.shape}")
print(f"y_train size: {y_train.shape}, y_test size: {y_test.shape}")

Training size: (80, 99), Test size: (20, 99)
y_train size: (80,), y_test size: (20,)


In [3]:

n_train, length = X_train.shape
n_test, _ = X_test.shape

time_vector = np.linspace(0, 1, length).reshape(1, length, 1)
time_channel_train = np.tile(time_vector, (n_train, 1, 1))
time_channel_test = np.tile(time_vector, (n_test, 1, 1))

X_train_augmented = np.concatenate([time_channel_train, X_train.reshape(n_train, length, 1)], axis=2)
X_test_augmented = np.concatenate([time_channel_test, X_test.reshape(n_test, length, 1)], axis=2)

print(f"Augmented training data shape: {X_train_augmented.shape}")
print(f"Augmented test data shape: {X_test_augmented.shape}")

Augmented training data shape: (80, 99, 2)
Augmented test data shape: (20, 99, 2)


## 4. Train NW Signature Regressor

We use cross-validation to automatically select the best bandwidth parameter.

In [4]:
# Extract initial conditions
#X0_train = X_train_augmented[:, 0, :]
#X0_test = X_test_augmented[:, 0, :]

X0_train = np.zeros_like(X_train_augmented[:, 0, :])
X0_test = np.zeros_like(X_test_augmented[:, 0, :])

# Initialize regressor with cross-validation
nw_regressor = NWSignatureRegressor(
    signature_level=3,
    input_type='paths',
    cv_folds=3,
    bandwidth_grid=np.linspace(0.001, 10, 50)
)

# Fit the model
print("Training NW Signature regressor...")
nw_regressor.fit(X_train_augmented, X0_train, y_train)
print(f"Best bandwidth: {nw_regressor.best_bandwidth_:.4f}")

Training NW Signature regressor...
Fitting NW estimator with input_type='paths'
Input shape: (80, 99, 2), target shape: (80,)
Target statistics: mean=-0.234227, std=0.838000, range=[-2.431745, 2.011835]
Computing signatures from paths...
Starting cross-validation for bandwidth selection...
Cross-validation: Testing 50 total parameter combinations
  Signature levels: 1
  Bandwidths: 50
Testing signature level 3...
Computed signatures shape: (80, 14)
    New best: level=3, bandwidth=0.001, score=0.890778
    New best: level=3, bandwidth=0.4091224489795919, score=0.799981
    New best: level=3, bandwidth=0.6131836734693878, score=0.767198
    New best: level=3, bandwidth=0.8172448979591838, score=0.741019
    New best: level=3, bandwidth=1.0213061224489797, score=0.721659
    New best: level=3, bandwidth=1.2253673469387756, score=0.708803
    New best: level=3, bandwidth=3.061918367346939, score=0.708623
    New best: level=3, bandwidth=3.265979591836735, score=0.707037
    New best: leve

In [5]:
# Make predictions
y_pred_nw = nw_regressor.predict(X_test_augmented, X0_test)

# Evaluate performance
mse_nw = mean_squared_error(y_test, y_pred_nw)
rmse_nw = np.sqrt(mse_nw)
mae_nw = mean_absolute_error(y_test, y_pred_nw)
r2_nw = r2_score(y_test, y_pred_nw)

print(f"\nNW Signature Test Performance:")
print(f"  MSE:  {mse_nw:.4f}")
print(f"  RMSE: {rmse_nw:.4f}")
print(f"  MAE:  {mae_nw:.4f}")
print(f"  R²:   {r2_nw:.4f}")


NW Signature Test Performance:
  MSE:  0.7923
  RMSE: 0.8901
  MAE:  0.6953
  R²:   -0.0253


## 4.bis Train NW EFM Signature Regressor

In [6]:
# Extract initial conditions
#X0_train = X_train_augmented[:, 0, :]
#X0_test = X_test_augmented[:, 0, :]

X0_train = np.zeros_like(X_train_augmented[:, 0, :])
X0_test = np.zeros_like(X_test_augmented[:, 0, :])

# Initialize regressor with cross-validation
nw_regressor = NWSignatureRegressor(
    signature_level=3,
    input_type='paths',
    cv_folds=3,
    efm_signature=True,
    bandwidth_grid=np.linspace(0.001, 10, 50),
    lam = np.array([0.1])
)

# Fit the model
print("Training NW Signature regressor...")
nw_regressor.fit(X_train_augmented, X0_train, y_train)
print(f"Best bandwidth: {nw_regressor.best_bandwidth_:.4f}")

Training NW Signature regressor...
Fitting NW estimator with input_type='paths'
Input shape: (80, 99, 2), target shape: (80,)
Target statistics: mean=-0.234227, std=0.838000, range=[-2.431745, 2.011835]
Computing signatures from paths...
Starting cross-validation for bandwidth selection...
Cross-validation: Testing 50 total parameter combinations
  Signature levels: 1
  Bandwidths: 50
Testing signature level 3...
Computed signatures shape: (80, 99)
    New best: level=3, bandwidth=0.001, score=0.762709
    New best: level=3, bandwidth=0.4091224489795919, score=0.762709

CV Summary: 150/150 successful attempts
Best CV MSE: 0.762709
Cross-validation completed. Best bandwidth: 0.4091224489795919
Training completed. Final training data shape: torch.Size([80, 99])
Using bandwidth: 0.4091224489795919, signature level: 3
Best bandwidth: 0.4091


In [7]:
# Make predictions
y_pred_nw = nw_regressor.predict(X_test_augmented, X0_test)

# Evaluate performance
mse_nw = mean_squared_error(y_test, y_pred_nw)
rmse_nw = np.sqrt(mse_nw)
mae_nw = mean_absolute_error(y_test, y_pred_nw)
r2_nw = r2_score(y_test, y_pred_nw)

print(f"\nNW Signature Test Performance:")
print(f"  MSE:  {mse_nw:.4f}")
print(f"  RMSE: {rmse_nw:.4f}")
print(f"  MAE:  {mae_nw:.4f}")
print(f"  R²:   {r2_nw:.4f}")


NW Signature Test Performance:
  MSE:  0.8151
  RMSE: 0.9028
  MAE:  0.7121
  R²:   -0.0548


### 5.1 DTW with Dynamic Time Warping (1-NN DTWD)

In [31]:
# DTW-based regressor
print("Training 1-NN DTW regressor...")
dtw_regressor = KNeighborsTimeSeriesRegressor(n_neighbors=1, distance="dtw")
dtw_regressor.fit(X_train, y_train)

# Predict
y_pred_dtw = dtw_regressor.predict(X_test)
mse_dtw = mean_squared_error(y_test, y_pred_dtw)
rmse_dtw = np.sqrt(mse_dtw)
r2_dtw = r2_score(y_test, y_pred_dtw)

print(f"DTW Test Performance:")
print(f"  MSE:  {mse_dtw:.4f}")
print(f"  RMSE: {rmse_dtw:.4f}")
print(f"  R²:   {r2_dtw:.4f}")

Training 1-NN DTW regressor...
DTW Test Performance:
  MSE:  1.6895
  RMSE: 1.2998
  R²:   -0.4144


### 5.2 DTWA (DTW with Averaging/Weighting)

In [32]:
# DTWA-based regressor (weighted DTW)
print("Training 1-NN DTWA regressor...")
dtwa_regressor = KNeighborsTimeSeriesRegressor(n_neighbors=1, distance="wdtw")
dtwa_regressor.fit(X_train, y_train)

# Predict
y_pred_dtwa = dtwa_regressor.predict(X_test)
mse_dtwa = mean_squared_error(y_test, y_pred_dtwa)
rmse_dtwa = np.sqrt(mse_dtwa)
r2_dtwa = r2_score(y_test, y_pred_dtwa)

print(f"DTWA Test Performance:")
print(f"  MSE:  {mse_dtwa:.4f}")
print(f"  RMSE: {rmse_dtwa:.4f}")
print(f"  R²:   {r2_dtwa:.4f}")

Training 1-NN DTWA regressor...
DTWA Test Performance:
  MSE:  2.1977
  RMSE: 1.4825
  R²:   -0.8399


### 5.3 Signature + Random Forest (SigP)

In [35]:
print(X_train.shape)

(80, 99)


In [43]:
# Signature-based features with Random Forest
import iisignature
print("Training Signature + Random Forest regressor...")



def compute_signature(X, level=3):
    n_samples, length = X.shape
    signatures = []
    for i in range(n_samples):
        signatures.append(iisignature.sig(X[i,:].reshape(-1,1), level))
    return np.array(signatures)

# Transform data
print(f"Raw data shape: {X_train.shape}")
X_train_sig = compute_signature(X_train)
X_test_sig = compute_signature(X_test)
print(f"Signature shape: {X_train_sig.shape}")

# Train Random Forest
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_sig, y_train)

# Predict
y_pred_sigp = rf_regressor.predict(X_test_sig)
mse_sigp = mean_squared_error(y_test, y_pred_sigp)
rmse_sigp = np.sqrt(mse_sigp)
r2_sigp = r2_score(y_test, y_pred_sigp)

print(f"Signature+RF Test Performance:")
print(f"  MSE:  {mse_sigp:.4f}")
print(f"  RMSE: {rmse_sigp:.4f}")
print(f"  R²:   {r2_sigp:.4f}")

Training Signature + Random Forest regressor...
Raw data shape: (80, 99)
Signature shape: (80, 3)
Signature+RF Test Performance:
  MSE:  1.9289
  RMSE: 1.3888
  R²:   -0.6149


### 5.4 Summary of Single Dataset Comparison

In [35]:
# Comparison summary
print("\n" + "="*80)
print(f"COMPARISON ON {dataset_name}")
print("="*80)
print(f"{'Method':<30} {'MSE':<15} {'RMSE':<15} {'R²':<10}")
print("-"*80)
print(f"{'NW Signature (Ours)':<30} {mse_nw:<15.4f} {rmse_nw:<15.4f} {r2_nw:<10.4f}")
print(f"{'1-NN DTW':<30} {mse_dtw:<15.4f} {rmse_dtw:<15.4f} {r2_dtw:<10.4f}")
print(f"{'1-NN DTWA':<30} {mse_dtwa:<15.4f} {rmse_dtwa:<15.4f} {r2_dtwa:<10.4f}")
print(f"{'Signature + Random Forest':<30} {mse_sigp:<15.4f} {rmse_sigp:<15.4f} {r2_sigp:<10.4f}")


COMPARISON ON Covid3Month
Method                         MSE             RMSE            R²        
--------------------------------------------------------------------------------
NW Signature (Ours)            0.0020          0.0447          -0.0024   
1-NN DTW                       0.0031          0.0557          -0.5576   
1-NN DTWA                      0.0025          0.0499          -0.2488   
Signature + Random Forest      0.0024          0.0492          -0.2171   


## 6. Batch Evaluation on Multiple Datasets

Evaluate all methods on several benchmark datasets.

In [28]:
# Select regression datasets for demonstration
datasets = [
    "Covid3Month",
    "AppliancesEnergy",
    #"BeijingPM10Quality",
    #"BeijingPM25Quality",
    #"BIDMC32HR"
]

In [29]:
# Run all methods on multiple datasets
results = {}

for dataset_name in datasets:
    print(f"\n{'='*60}")
    print(f"Processing: {dataset_name}")
    print('='*60)
    
    try:
        # Load data
        X_train, y_train = load_regression(dataset_name, split="train")
        X_test, y_test = load_regression(dataset_name, split="test")
        
        # Shuffle
        X_train, y_train = shuffle(X_train, y_train, random_state=42)
        X_test, y_test = shuffle(X_test, y_test, random_state=42)
        
        print(f"Data shape: {X_train.shape}")
        print(f"Target range: [{y_train.min():.2f}, {y_train.max():.2f}]")
        
        results[dataset_name] = {}
        
        # 1. NW Signature Regressor
        try:
            X_train_augmented = augment_with_time(X_train)
            X_test_augmented = augment_with_time(X_test)

            # Scale Train data
            X_train_augmented = scaler.fit_transform(
                X_train_augmented.reshape(-1, X_train_augmented.shape[-1])
            ).reshape(X_train_augmented.shape)

            # Scale Test data
            X_test_augmented = scaler.fit_transform(
                X_test_augmented.reshape(-1, X_test_augmented.shape[-1])
            ).reshape(X_test_augmented.shape)

            # Extract initial conditions
            X0_train = X_train_augmented[:, 0, :]
            X0_test = X_test_augmented[:, 0, :]
            
            nw_reg = NWSignatureRegressor(
                signature_level=3,
                input_type='paths',
                cv_folds=3,
                bandwidth_grid=np.linspace(0.01, 2.0, 30)
            )
            nw_reg.fit(X_train_augmented, X0_train, y_train)
            y_pred = nw_reg.predict(X_test_augmented,X0_test)
            
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)
            
            results[dataset_name]['NW Signature'] = {'mse': mse, 'rmse': rmse, 'r2': r2}
            print(f"  NW Signature - MSE: {mse:.4f}, R²: {r2:.4f}")
        except Exception as e:
            print(f"  NW Signature failed: {str(e)}")
            results[dataset_name]['NW Signature'] = None
        
        # 2. DTW
        try:
            dtw_reg = KNeighborsTimeSeriesRegressor(n_neighbors=1, distance="dtw")
            dtw_reg.fit(X_train, y_train)
            y_pred = dtw_reg.predict(X_test)
            
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)
            
            results[dataset_name]['DTW'] = {'mse': mse, 'rmse': rmse, 'r2': r2}
            print(f"  DTW - MSE: {mse:.4f}, R²: {r2:.4f}")
        except Exception as e:
            print(f"  DTW failed: {str(e)}")
            results[dataset_name]['DTW'] = None
        
        # 3. DTWA
        try:
            dtwa_reg = KNeighborsTimeSeriesRegressor(n_neighbors=1, distance="wdtw")
            dtwa_reg.fit(X_train, y_train)
            y_pred = dtwa_reg.predict(X_test)
            
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)
            
            results[dataset_name]['DTWA'] = {'mse': mse, 'rmse': rmse, 'r2': r2}
            print(f"  DTWA - MSE: {mse:.4f}, R²: {r2:.4f}")
        except Exception as e:
            print(f"  DTWA failed: {str(e)}")
            results[dataset_name]['DTWA'] = None
        
        # 4. Signature + Random Forest
        try:
            sig_transform = SignatureTransformer(
                augmentation_list=("addtime",),
                window_name="global",
                sig_tfm="signature",
                depth=3,
            )
            X_train_sig = sig_transform.fit_transform(X_train)
            X_test_sig = sig_transform.transform(X_test)
            
            rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
            rf_reg.fit(X_train_sig, y_train)
            y_pred = rf_reg.predict(X_test_sig)
            
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)
            
            results[dataset_name]['Sig+RF'] = {'mse': mse, 'rmse': rmse, 'r2': r2}
            print(f"  Sig+RF - MSE: {mse:.4f}, R²: {r2:.4f}")
        except Exception as e:
            print(f"  Sig+RF failed: {str(e)}")
            results[dataset_name]['Sig+RF'] = None
        
    except Exception as e:
        print(f"Error loading {dataset_name}: {str(e)}")
        results[dataset_name] = {'NW Signature': None, 'DTW': None, 'DTWA': None, 'Sig+RF': None}


Processing: Covid3Month
Data shape: (140, 1, 84)
Target range: [0.00, 0.18]
Fitting NW estimator with input_type='paths'
Input shape: (140, 84, 2), target shape: (140,)
Target statistics: mean=0.036898, std=0.040208, range=[0.000000, 0.176471]
Computing signatures from paths...
Starting cross-validation for bandwidth selection...
Cross-validation: Testing 30 total parameter combinations
  Signature levels: 1
  Bandwidths: 30
Testing signature level 3...
Computed signatures shape: (140, 14)
    New best: level=3, bandwidth=0.01, score=0.001605

CV Summary: 90/90 successful attempts
Best CV MSE: 0.001605
Cross-validation completed. Best bandwidth: 0.01
Training completed. Final training data shape: torch.Size([140, 14])
Using bandwidth: 0.01, signature level: 3
  NW Signature - MSE: 0.0018, R²: 0.1198
  DTW - MSE: 0.0031, R²: -0.5576
  DTWA - MSE: 0.0025, R²: -0.2488
  Sig+RF - MSE: 0.0024, R²: -0.2171

Processing: AppliancesEnergy
Data shape: (95, 24, 144)
Target range: [5.34, 26.28]
F